# FLake: Colab preview

Try the new interface before replacing the public GitHub version. The MSI preview notebook prepares **FLake_Century_Colab_Preview.zip**, containing three real lakes’ historical inputs and all four SSPs through 2099.

Run Setup, upload that small ZIP when prompted, then run Open FLake. All scientific controls and plots are the same as the full version; only the selectable lake subset is smaller. The full 7.19 GiB dataset is not uploaded to Colab.

## Setup

Loads the model and notebook dependencies. Weather is downloaded only when needed for your selected lake and scenario. The first run may download approximately two 100 MiB bundles; subsequent runs reuse cached inputs.

In [ ]:
# Public repository mode. The separate preview notebook sets this to True.
USE_PREVIEW_UPLOAD = True

from pathlib import Path
import sys, subprocess, tempfile, zipfile, importlib

try:
    from google.colab import files, output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def unpack_safely(archive_path, destination):
    with zipfile.ZipFile(archive_path) as archive:
        for name in archive.namelist():
            path = Path(name)
            if path.is_absolute() or ".." in path.parts:
                raise ValueError("Unsafe ZIP member.")
        archive.extractall(destination)

if USE_PREVIEW_UPLOAD:
    if not IN_COLAB:
        raise RuntimeError("Open this preview notebook in Google Colab, then upload the small preview ZIP prepared on MSI.")
    print("Upload FLake_Century_Colab_Preview.zip (the small three-lake preview, not the 7 GiB dataset).")
    uploaded = files.upload()
    names = [name for name in uploaded if name.endswith(".zip")]
    if len(names) != 1:
        raise ValueError("Upload exactly one preview ZIP.")
    location = Path(tempfile.mkdtemp(prefix="flake_preview_"))
    unpack_safely(names[0], location)
    PROJECT = location / "flake-python"
else:
    candidates = [Path.cwd(), Path.cwd() / "flake-explorer", Path("/content/flake-explorer")]
    PROJECT = next((p for p in candidates if (p / "flake/dashboard.py").is_file()), None)
    if PROJECT is None:
        if not IN_COLAB:
            raise FileNotFoundError("Open this notebook inside the extracted flake-explorer folder.")
        PROJECT = Path(tempfile.mkdtemp(prefix="flake_python_")) / "repository"
        completed = subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", "main",
            "https://github.com/Sandesh710/flake-explorer.git", str(PROJECT)], stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
        if completed.returncode:
            raise RuntimeError("Could not load the public repository. Check GitHub access and rerun setup. " + completed.stderr[-500:])
    if not (PROJECT / "data/data_manifest.json").is_file():
        raise RuntimeError("The century dashboard code has not been uploaded to GitHub yet. Use Preview_FLake_Colab.ipynb with the small preview ZIP until publication.")

if IN_COLAB:
    print("Preparing the notebook dependencies…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "-r", str(PROJECT / "requirements.txt")])
    output.enable_custom_widget_manager()
for name in list(sys.modules):
    if name == "flake" or name.startswith("flake.") or name == "preview_check":
        sys.modules.pop(name)
sys.path.insert(0, str(PROJECT))
importlib.invalidate_caches()
LOCAL_EXPORT = PROJECT / "preview_inputs" if (PROJECT / "preview_inputs").is_dir() else None
print("Ready. Run the next cell to open FLake.")


## Open FLake

Select a lake on the map or search by name, county or ID. Keep its defaults for a first run. Initial conditions apply in 1950; warm-up precedes the displayed 1995–2099 results. Year selectors redraw figures without rerunning the model.

In [ ]:
from flake.dashboard import launch

dashboard = launch(root=PROJECT, local_export=LOCAL_EXPORT)


FLake was developed by Dmitrii Mironov and collaborators at the German Weather Service (DWD). The Python engine retains its original source notices. Model reference: [Mironov (2008), COSMO Technical Report 11](https://doi.org/10.5676/DWD_pub/nwv/cosmo-tr_11).

The temperature–depth section reconstructs FLake’s assumed bulk profile. Meteorology: GFDL-ESM4 from NEX-GDDP-CMIP6. See the repository [user guide](https://github.com/Sandesh710/flake-explorer/blob/main/docs/USER_GUIDE.md), [attribution](https://github.com/Sandesh710/flake-explorer/blob/main/NOTICE.md) and [citation](https://github.com/Sandesh710/flake-explorer/blob/main/CITATION.cff).